# UHPC: Group-Aware Splitting & Leave-One-Publication-Out (LOPO) Validation

This notebook evaluates boosting models on UHPC data using publication-aware validation:
- GroupShuffleSplit (70% Train / 15% Val / 15% Test) to prevent data leakage across papers
- Dynamic OneHot and TargetEncoding feature pipeline (via pipeline_w8.py)
- GridSearchCV hyperparameter tuning for XGBoost, HistGradientBoosting, and AdaBoost
- Leave-One-Publication-Out (LOPO) cross-validation for major publications (>= 50 samples)

# UHPC

## Data Loading & Feature Isolation


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, LeaveOneGroupOut, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from pipeline import preprocessing_pipeline

# 50-feature dataset
df_drop = pd.read_csv(".//initial_data_preparation/Datasets/processed/uhpc_dataset/semantic_recoding_features_50.csv")
# df_drop = pd.read_csv(".//initial_data_preparation/Datasets/processed/uhpc_dataset/semantic_recoding_features_20.csv")

# Apply drops 
df_drop = df_drop.drop(columns=['cement_grade'])
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_drop[fiber_cols] = df_drop[fiber_cols].fillna(0) 

groups = df_drop['publication']

# Proceed to split data
Y = df_drop['cs_28d'] 
X = df_drop.drop(columns=['cs_28d', 'publication'])

### Identify columns for encoding

In [3]:
# Identify all categorical columns
str_cols = X.select_dtypes(include=['object', 'string']).columns

# Sort text columns into One-Hot (<= 10 categories) and Target Encoding (> 10 categories)
one_hot_encode_cols = str_cols[X[str_cols].nunique() <= 10].tolist()
target_encode_cols = str_cols[X[str_cols].nunique() > 10].tolist()

# Identify all numeric columns
numeric_features = X.select_dtypes(include='number').columns.tolist()

# Print them out to verify
print(f"One-Hot columns: {len(one_hot_encode_cols)}")
print(f"Target Encoded columns: {len(target_encode_cols)}")
print(f"Numeric columns: {len(numeric_features)}")

One-Hot columns: 5
Target Encoded columns: 4
Numeric columns: 24


## Split: Train, Validation, Test

In [4]:
# First split - Separate Test (15%) from the rest (85%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss1.split(X, Y, groups=groups))

X_trainval = X.iloc[trainval_idx]
Y_trainval = Y.iloc[trainval_idx]
groups_trainval = groups.iloc[trainval_idx]

X_test = X.iloc[test_idx]
Y_test = Y.iloc[test_idx]

# Second split - Separate Validation (15%) from Train (70%) 
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=42)
train_idx, val_idx = next(gss2.split(X_trainval, Y_trainval, groups=groups_trainval))

X_train = X_trainval.iloc[train_idx]
Y_train = Y_trainval.iloc[train_idx]

X_val = X_trainval.iloc[val_idx]
Y_val = Y_trainval.iloc[val_idx]

print(f"Train rows: {len(X_train)}")
print(f"Validation rows: {len(X_val)}")
print(f"Test rows: {len(X_test)}")
print(f"Total: {len(X_train) + len(X_val) + len(X_test)}")

Train rows: 1459
Validation rows: 331
Test rows: 283
Total: 2073


## Applying GridSearchCV 

In [ ]:

# Define the base models
models = {
    'XGBoost': XGBRegressor(random_state=42),
    'HistGradient': HistGradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42)
}

# Define the parameter grids. 
param_grids = {
    'XGBoost': {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [3, 5, 10],
        'model__learning_rate': [0.05, 0.1, 0.2]
    },
    'HistGradient': {
        'model__max_iter': [50, 100, 200],
        'model__max_depth': [3, 5, 10],
        'model__learning_rate': [0.05, 0.1, 0.2]
    },
    'AdaBoost': {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.05, 0.1, 0.2]
    }
}

best_pipelines = {}
tuning_results = []

for name, model in models.items():
    print(f"\n--- Tuning {name} ---")
    
    # Build your custom pipeline
    pipe = preprocessing_pipeline(one_hot_encode_cols, target_encode_cols, numeric_features, estimator=model)
    
    # Run GridSearch
    gs = GridSearchCV(pipe, param_grids[name], cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    gs.fit(X_train, Y_train)
    
    # Save the best pipeline for later use
    best_pipelines[name] = gs.best_estimator_
    
    # Evaluate it on the completely separate Validation set
    val_Y_pred = gs.best_estimator_.predict(X_val)
    val_r2 = r2_score(Y_val, val_Y_pred)
    val_rmse = root_mean_squared_error(Y_val, val_Y_pred)
    val_mae = mean_absolute_error(Y_val, val_Y_pred)
    
    tuning_results.append({
        'Model': name,
        'Best Params': gs.best_params_,
        'Val R2': val_r2,
        'Val MAE': val_mae,
        'Val RMSE': val_rmse
    })
    
    print(f"Validation R2: {val_r2:.4f}, MAE {val_mae:.4f} RMSE: {val_rmse:.4f}")

tuning_df = pd.DataFrame(tuning_results)[['Model', 'Val R2', 'Val MAE', 'Val RMSE']]
display(tuning_df)


--- Tuning XGBoost ---
Validation R2: 0.3732, MAE 28.4452 RMSE: 38.9367

--- Tuning HistGradient ---
Validation R2: 0.3607, MAE 29.5990 RMSE: 39.3233

--- Tuning AdaBoost ---
Validation R2: 0.3655, MAE 29.2272 RMSE: 39.1745


,Model,Val R2,Val MAE,Val RMSE
0,XGBoost,0.373188,28.445171,38.936748
1,HistGradient,0.360681,29.599050,39.323289
2,AdaBoost,0.365510,29.227187,39.174509


## Leave one publication group out

In [ ]:
# Find publications with >= 50 rows
publ_counts = groups.value_counts()
eligible_pubs = publ_counts[publ_counts >= 50].index.tolist()
print(f"Eligible publications (>=50 rows): {eligible_pubs}")

lopo_results = []

# Leave-One-Out Loop
for publ_name in eligible_pubs:
    
    # Create masks to isolate the specific publication
    test_mask = groups == publ_name
    train_mask = groups != publ_name
    X_train_lopo = X[train_mask]
    X_test_lopo = X[test_mask]
    Y_train_lopo = Y[train_mask]
    Y_test_lopo = Y[test_mask]
    
    # Predict using best pipelines
    for name, pipe in best_pipelines.items():
        
        # Fit the best tuned pipeline on the training dataw
        pipe.fit(X_train_lopo, Y_train_lopo)
        Y_pred = pipe.predict(X_test_lopo)
        
        r2 = r2_score(Y_test_lopo, Y_pred)
        rmse = root_mean_squared_error(Y_test_lopo, Y_pred)
        mae = mean_absolute_error(Y_test_lopo, Y_pred)
        
        residuals = Y_test_lopo.values - Y_pred
        mean_residual = np.mean(residuals)
        direction = "Over-predicting" if mean_residual < 0 else "Under-predicting"
        worst_error = np.max(np.abs(residuals))
        
        lopo_results.append({
            'Publication': publ_name,
            'Model': name,
            'Rows': test_mask.sum(),
            'R2': r2,
            'MAE': mae,
            'RMSE': rmse,
            'Direction': direction,
            'Worst Error (MPa)': round(worst_error, 2)
        })
        
        print(f"{publ_name} | {name}: R2={r2:.4f}, RMSE={rmse:.4f}, {direction}")

lopo_df = pd.DataFrame(lopo_results)
display(lopo_df)

Eligible publications (>=50 rows): ['Ref-144-Research', 'Ref-121-Research', 'Ref-141-Research', 'Ref-48-Research', 'Ref-85-Research', 'Ref-139-Research']
Ref-144-Research | XGBoost: R2=0.4520, RMSE=22.1578, Under-predicting
Ref-144-Research | HistGradient: R2=0.5058, RMSE=21.0420, Over-predicting
Ref-144-Research | AdaBoost: R2=0.2651, RMSE=25.6593, Over-predicting
Ref-121-Research | XGBoost: R2=0.5450, RMSE=19.6093, Under-predicting
Ref-121-Research | HistGradient: R2=0.3112, RMSE=24.1253, Over-predicting
Ref-121-Research | AdaBoost: R2=0.1498, RMSE=26.8043, Over-predicting
Ref-141-Research | XGBoost: R2=-0.4118, RMSE=14.8363, Over-predicting
Ref-141-Research | HistGradient: R2=0.0710, RMSE=12.0352, Over-predicting
Ref-141-Research | AdaBoost: R2=-0.5809, RMSE=15.7000, Over-predicting
Ref-48-Research | XGBoost: R2=-0.0547, RMSE=29.0422, Under-predicting
Ref-48-Research | HistGradient: R2=-0.0665, RMSE=29.2052, Under-predicting
Ref-48-Research | AdaBoost: R2=-0.4756, RMSE=34.3521, Unde

,Publication,Model,Rows,R2,MAE,RMSE,Direction,Worst Error (MPa)
0,Ref-144-Research,XGBoost,112,0.451979,17.688931,22.157762,Under-predicting,54.87
1,Ref-144-Research,HistGradient,112,0.505779,17.167189,21.042042,Over-predicting,58.42
2,Ref-144-Research,AdaBoost,112,0.265089,19.645466,25.659296,Over-predicting,62.09
3,Ref-121-Research,XGBoost,80,0.544954,16.263481,19.609261,Under-predicting,50.34
4,Ref-121-Research,HistGradient,80,0.311220,19.826547,24.125343,Over-predicting,63.85
5,Ref-121-Research,AdaBoost,80,0.149756,22.042349,26.804347,Over-predicting,59.68
6,Ref-141-Research,XGBoost,73,-0.411769,11.574594,14.836287,Over-predicting,45.10
7,Ref-141-Research,HistGradient,73,0.070991,8.612774,12.035200,Over-predicting,39.97
8,Ref-141-Research,AdaBoost,73,-0.580923,13.920799,15.699968,Over-predicting,33.87
9,Ref-48-Research,XGBoost,72,-0.054659,24.898211,29.042185,Under-predicting,60.14
